# Fraud / Anomaly Detection
## Notebook 10 — DBSCAN Anomaly Detection

### Objective
Treat low-density DBSCAN noise points as anomalies.

## Imports and sample

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.fraud_detection import load_features_labels, evaluate_scores, save_results
X, y = load_features_labels(ROOT)
RANDOM_STATE = 42
print('Features:', X.shape, '| fraud:', int(y.sum()))
from src.fraud_detection import stratified_working_sample
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
Xw, yw = stratified_working_sample(X, y, max_rows=12000, random_state=RANDOM_STATE)

**Interpretation.** DBSCAN is sensitive to scale, dimension, and memory use.

**ML decision.** Use a bounded sample and estimate epsilon from neighbor distances.

## Estimate epsilon and fit

In [ ]:
nn = NearestNeighbors(n_neighbors=10, n_jobs=-1).fit(Xw)
distances, _ = nn.kneighbors(Xw)
kdist = distances[:, -1]
eps = float(np.quantile(kdist, .95))
labels = DBSCAN(eps=eps, min_samples=10, n_jobs=-1).fit_predict(Xw)
scores = kdist
result = evaluate_scores(yw, scores, 'DBSCAN k-distance')
result.update({'eps': eps, 'noise_points': int((labels == -1).sum())})
pd.DataFrame({'row_index': Xw.index, 'label': yw, 'score': scores, 'cluster': labels}).to_csv(ROOT/'reports/dbscan_scores.csv', index=False)
result

**Interpretation.** K-neighbor distance provides a continuous density anomaly score; cluster -1 is DBSCAN's hard noise assignment.

**ML decision.** Use the continuous score for metrics and the hard label for cluster interpretation.